# Tutorial 25: MCP and A2A Protocol

In this final tutorial we cover two open protocols that define how modern AI agents interact with the world: the **Model Context Protocol (MCP)** for agent-to-tool communication, and the **Agent-to-Agent (A2A) Protocol** for cross-framework agent interoperability.

## 1. The Interoperability Stack

```
Agent A (LangGraph) ←──A2A──→ Agent B (CrewAI / AutoGen / any framework)
     │
    MCP
     │
Tool servers (filesystem, databases, APIs, browsers...)
```

| Protocol | Governs | Created by |
|---|---|---|
| **MCP** | Agent ↔ Tool communication | Anthropic (open standard) |
| **A2A** | Agent ↔ Agent communication | Google → Linux Foundation |

MCP standardises how an agent discovers and calls tools. A2A standardises how agents from different frameworks communicate with each other.

## 2. Setup

In [1]:
import os
import warnings
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

warnings.filterwarnings("ignore", category=DeprecationWarning)

if os.getenv("GROQ_API_KEY"):
    model = init_chat_model("groq:llama-3.1-8b-instant", temperature=0)
    print("Setup complete. Groq model ready.")
else:
    model = None
    print("Setup complete. GROQ_API_KEY is not set, so model calls are skipped.")


Setup complete. GROQ_API_KEY is not set, so model calls are skipped.


## 3. Model Context Protocol (MCP)

LangChain 1.4 moved MCP into `langchain.mcp`. `MCPAdapter` replaces `MultiServerMCPClient` from `langchain-mcp-adapters`.

The adapter accepts an HTTP URL, a local script path, an in-process FastMCP server, or a config dict with several servers. `await adapter.list_tools()` returns ordinary LangChain tools for `create_agent`.

The `langchain.mcp` namespace is still beta. Importing it raises `LangChainBetaWarning`.


In [2]:
print(
"""
MCP with LangChain 1.4+:

  create_agent
       |
  MCPAdapter.list_tools()
       |
  in-process FastMCP server, a stdio script, or an https URL
       |
  LangChain tools
"""
)



MCP with LangChain 1.4+:

  create_agent
       |
  MCPAdapter.list_tools()
       |
  in-process FastMCP server, a stdio script, or an https URL
       |
  LangChain tools



## 4. An in-process MCP server

An in-process FastMCP server needs no subprocess and no socket, so this cell runs as written. The same `MCPAdapter` accepts a `Path` to a script (stdio) or an `https://` URL (streamable HTTP). A plain string must be a URL. A string that happens to name a local file is rejected, so a path from configuration cannot launch a program by accident.


In [3]:
import warnings
from fastmcp import FastMCP
from langchain.mcp import MCPAdapter

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message=".*in beta.*")

server = FastMCP("arithmetic")


@server.tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b


async def demo_mcp():
    async with MCPAdapter(server) as adapter:
        tools = await adapter.list_tools()
        print("MCP tools:", [tool.name for tool in tools])
        result = await tools[0].ainvoke({"a": 6, "b": 7})
        text = result if isinstance(result, str) else result[0]["text"]
        print("multiply(6, 7) =", text)
        assert text == "42"
        if model is None:
            print("Skipped the agent call. Set GROQ_API_KEY to let a model use the MCP tool.")
            return
        agent = create_agent(model, tools)
        answer = await agent.ainvoke({
            "messages": [HumanMessage(content="Use the multiply tool. What is 6 times 7?")]
        })
        print(answer["messages"][-1].content[:200])


await demo_mcp()


MCP tools: ['multiply']
multiply(6, 7) = 42
Skipped the agent call. Set GROQ_API_KEY to let a model use the MCP tool.


## 5. The same tools without a server

The tools above are normal LangChain tools after `list_tools()`. Hand-written tools are the same shape. This section keeps a small filesystem example that does not need MCP at all.


In [4]:
import os
import glob
from langchain_core.tools import tool

# These mirror the tools exposed by @modelcontextprotocol/server-filesystem

@tool
def mcp_read_file(path: str) -> str:
    """Read the contents of a file at the given path."""
    try:
        with open(path, 'r') as f:
            return f.read()
    except FileNotFoundError:
        return f"Error: File not found at {path}"
    except Exception as e:
        return f"Error reading file: {str(e)}"


@tool
def mcp_write_file(path: str, content: str) -> str:
    """Write content to a file at the given path."""
    try:
        with open(path, 'w') as f:
            f.write(content)
        return f"Successfully wrote {len(content)} characters to {path}"
    except Exception as e:
        return f"Error writing file: {str(e)}"


@tool
def mcp_list_directory(path: str) -> str:
    """List files and directories at the given path."""
    try:
        items = os.listdir(path)
        result = []
        for item in sorted(items)[:20]:  # limit to 20 items
            full_path = os.path.join(path, item)
            item_type = "dir" if os.path.isdir(full_path) else "file"
            result.append(f"{item_type}: {item}")
        return "\n".join(result) if result else "Empty directory"
    except Exception as e:
        return f"Error listing directory: {str(e)}"


@tool
def mcp_search_files(path: str, pattern: str) -> str:
    """Search for files matching a pattern in the given directory."""
    try:
        matches = glob.glob(os.path.join(path, "**", pattern), recursive=True)[:10]
        return "\n".join(matches) if matches else f"No files matching '{pattern}' found in {path}"
    except Exception as e:
        return f"Error searching: {str(e)}"


mcp_tools = [mcp_read_file, mcp_write_file, mcp_list_directory, mcp_search_files]
print(f"Simulated MCP filesystem tools: {[t.name for t in mcp_tools]}")

Simulated MCP filesystem tools: ['mcp_read_file', 'mcp_write_file', 'mcp_list_directory', 'mcp_search_files']


In [5]:
if model is None:
    print("Skipped the file agent. Set GROQ_API_KEY to run it.")
else:
    mcp_agent = create_agent(
        model=model,
        tools=mcp_tools,
        system_prompt=(
            "You are a file management assistant. Use the filesystem tools to help the user "
            "manage their files. Always confirm before writing or modifying files."
        ),
    )
    result = mcp_agent.invoke({
        "messages": [
            HumanMessage(
                content="Write /tmp/mcp_test.txt with the content Hello from MCP agent, then read it back."
            )
        ]
    })
    print(result["messages"][-1].content[:300])


Skipped the file agent. Set GROQ_API_KEY to run it.


## 6. Agent-to-Agent (A2A) Protocol

A2A defines how two agents — from different frameworks — communicate:

- Each agent exposes an **Agent Card** at `/.well-known/agent.json` describing its capabilities
- Communication uses JSON-RPC 2.0 over HTTP/SSE
- Three RPC methods: `message/send`, `message/stream`, `tasks/get`

This means a LangGraph agent can call a CrewAI agent (or vice versa) without knowing anything about the other framework.

In [6]:
# A2A Agent Card structure
agent_card_example = {
    "name": "Research Agent",
    "description": "Performs deep research on any topic and returns structured reports.",
    "url": "https://my-research-agent.example.com",
    "version": "1.0.0",
    "capabilities": {
        "streaming": True,
        "pushNotifications": False,
    },
    "defaultInputModes": ["text/plain"],
    "defaultOutputModes": ["text/plain", "application/json"],
    "skills": [
        {
            "id": "research",
            "name": "Deep Research",
            "description": "Research a topic thoroughly and return a structured report.",
            "inputModes": ["text/plain"],
            "outputModes": ["text/plain"],
        }
    ]
}

import json
print("A2A Agent Card example:")
print(json.dumps(agent_card_example, indent=2))

A2A Agent Card example:
{
  "name": "Research Agent",
  "description": "Performs deep research on any topic and returns structured reports.",
  "url": "https://my-research-agent.example.com",
  "version": "1.0.0",
  "capabilities": {
    "streaming": true,
    "pushNotifications": false
  },
  "defaultInputModes": [
    "text/plain"
  ],
  "defaultOutputModes": [
    "text/plain",
    "application/json"
  ],
  "skills": [
    {
      "id": "research",
      "name": "Deep Research",
      "description": "Research a topic thoroughly and return a structured report.",
      "inputModes": [
        "text/plain"
      ],
      "outputModes": [
        "text/plain"
      ]
    }
  ]
}


In [7]:
# A2A message/send request format
a2a_request_example = {
    "jsonrpc": "2.0",
    "id": "req-001",
    "method": "message/send",
    "params": {
        "message": {
            "role": "user",
            "parts": [
                {"kind": "text", "text": "Research the latest advances in quantum computing."}
            ],
            "messageId": "msg-001"
        },
        "configuration": {
            "acceptedOutputModes": ["text/plain"]
        }
    }
}

print("A2A message/send request format:")
print(json.dumps(a2a_request_example, indent=2))

A2A message/send request format:
{
  "jsonrpc": "2.0",
  "id": "req-001",
  "method": "message/send",
  "params": {
    "message": {
      "role": "user",
      "parts": [
        {
          "kind": "text",
          "text": "Research the latest advances in quantum computing."
        }
      ],
      "messageId": "msg-001"
    },
    "configuration": {
      "acceptedOutputModes": [
        "text/plain"
      ]
    }
  }
}


## 7. A2A with the Agent Server

There is no `LangGraphServer` class in the `langgraph` package. A compiled graph becomes an A2A server when the Agent Server runs it. `langgraph dev` serves each assistant at `/a2a/{assistant_id}` and publishes the card at `/.well-known/agent-card.json?assistant_id={assistant_id}`.

The graph state has to include `messages`. The assistant id is the key in `langgraph.json`.


In [8]:
print("""
langgraph.json
{
  "dependencies": ["."],
  "graphs": {
    "research": "./agent.py:graph"
  }
}

agent.py
  from langchain.agents import create_agent
  graph = create_agent(model, tools)

Start the server:
  langgraph dev

Agent Card:
  GET http://localhost:2024/.well-known/agent-card.json?assistant_id=research

Send a message:
  POST http://localhost:2024/a2a/research
  {"jsonrpc": "2.0", "id": 1, "method": "message/send", "params": {"message": {...}}}

Call another assistant by posting to its /a2a/{assistant_id} URL.
The local demo in the next cell uses plain functions so it runs without the CLI.
""")



langgraph.json
{
  "dependencies": ["."],
  "graphs": {
    "research": "./agent.py:graph"
  }
}

agent.py
  from langchain.agents import create_agent
  graph = create_agent(model, tools)

Start the server:
  langgraph dev

Agent Card:
  GET http://localhost:2024/.well-known/agent-card.json?assistant_id=research

Send a message:
  POST http://localhost:2024/a2a/research
  {"jsonrpc": "2.0", "id": 1, "method": "message/send", "params": {"message": {...}}}

Call another assistant by posting to its /a2a/{assistant_id} URL.
The local demo in the next cell uses plain functions so it runs without the CLI.



## 8. Multi-Provider Agent Network

With MCP and A2A together, you can build a network where each agent uses its strengths. The key insight is that the **orchestrator does not have to be an LLM** — it can be pure Python logic that routes work to specialist agents. This is more reliable and better demonstrates the A2A principle: agents are called like remote services.

In [9]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

# ── Specialist agents implemented as pure Python functions ────────────────────
# In real A2A each of these would be a full agent running on its own server
# (possibly a different framework entirely). For this demo we use Python
# functions that simulate what those agents would do — no API calls needed.

def data_processor_agent(task: str) -> str:
    """
    Simulated data-processor agent.
    Receives a task string, writes structured data to /tmp, returns a summary.
    """
    data = (
        "Tutorial Series Accomplishments\n"
        "================================\n"
        "1. LangChain 1.0 foundations: LCEL chains, agents, RAG, memory\n"
        "2. LangGraph patterns: HITL, Supervisor, Swarm, Subgraphs, Map-Reduce\n"
        "3. Advanced topics: Deep Agents, Middleware, MCP, A2A Protocol\n"
    )
    path = "/tmp/tutorial_data.txt"
    mcp_write_file.invoke({"path": path, "content": data})
    return f"Structured data written to {path}:\n{data}"


def report_writer_agent(task: str) -> str:
    """
    Simulated report-writer agent.
    Receives data in the task string, formats it as a polished report.
    """
    report = (
        "Summary Report — LangChain & LangGraph Tutorial Series\n"
        "=======================================================\n"
        "This tutorial series guides learners from LangChain basics through\n"
        "production-grade multi-agent systems. Key accomplishments include mastering\n"
        "LCEL, building stateful LangGraph workflows, implementing Human-in-the-Loop\n"
        "patterns, and deploying agents that communicate via the MCP and A2A protocols.\n"
        "The series uses Groq for fast, free LLM inference throughout.\n"
    )
    path = "/tmp/tutorial_report.txt"
    mcp_write_file.invoke({"path": path, "content": report})
    return report


# ── Python orchestrator — pure routing logic, no LLM ─────────────────────────
# The key A2A insight: the orchestrator treats each specialist as a black-box
# service it calls over a standard interface (here: a Python function;
# in production: an HTTP POST to /.well-known/agent.json endpoint).

def a2a_send(agent_fn, task: str) -> str:
    """Simulate an A2A message/send call."""
    return agent_fn(task)

def orchestrate(request: str) -> str:
    print(f"[orchestrator] Request received: {request[:70]}")

    print("[orchestrator] → A2A call to data_processor")
    data = a2a_send(data_processor_agent, "Produce structured data about the tutorial series")
    print(f"[data_processor] ✓ returned {len(data)} chars")

    print("[orchestrator] → A2A call to report_writer")
    report = a2a_send(report_writer_agent, f"Write a report from this data:\n{data[:300]}")
    print(f"[report_writer] ✓ returned {len(report)} chars")

    return report

final_report = orchestrate(
    "Process the data and write a summary report about our tutorial series."
)
print("\n=== FINAL REPORT ===")
print(final_report)

[orchestrator] Request received: Process the data and write a summary report about our tutorial series.
[orchestrator] → A2A call to data_processor
[data_processor] ✓ returned 312 chars
[orchestrator] → A2A call to report_writer
[report_writer] ✓ returned 471 chars

=== FINAL REPORT ===
Summary Report — LangChain & LangGraph Tutorial Series
This tutorial series guides learners from LangChain basics through
production-grade multi-agent systems. Key accomplishments include mastering
LCEL, building stateful LangGraph workflows, implementing Human-in-the-Loop
patterns, and deploying agents that communicate via the MCP and A2A protocols.
The series uses Groq for fast, free LLM inference throughout.



## 9. Choosing the Right Protocol

| Use case | Protocol |
|---|---|
| Agent calls a filesystem, database, or browser tool | MCP |
| Agent calls another agent from a different framework | A2A |
| Agent calls another LangGraph agent | LangGraph `RemoteGraph` or direct `.invoke()` |
| Agent shares tools within the same framework | Standard LangChain tools |

MCP and A2A are **complementary**: MCP for tool access, A2A for agent interoperability.

## 10. Conclusion

In this tutorial we covered:
- **MCP (Model Context Protocol)**: `MCPAdapter` turns an in-process, stdio, or HTTP server into LangChain tools
- **A2A Protocol**: standardised agent-to-agent communication, Agent Cards, JSON-RPC interface
- How both protocols enable truly provider-neutral, framework-agnostic agent networks

---

Here is a summary of the series so far:

| Tutorial | Topic |
|---|---|
| 1-6 | LangChain fundamentals: models, prompts, chains, agents, memory, RAG |
| 7-9 | LangGraph: graphs, complex flows, LangChain + LangGraph integration |
| 10-13 | Real-world apps, structured data, advanced LangChain, best practices |
| 14 | Human-in-the-Loop: `interrupt()`, `Command(resume=...)` |
| 15 | Supervisor Agent: `langgraph-supervisor`, hierarchical coordination |
| 16 | Swarm Agents: `langgraph-swarm`, peer-to-peer handoffs |
| 17 | Subgraphs: nested graphs, state sharing, `Command.PARENT` |
| 18 | Parallelization + Map-Reduce: `Send` API, fan-out/fan-in |
| 19 | Long-term Memory: `MemorySaver`, `SqliteSaver`, `InMemoryStore` |
| 20 | Time Travel: `get_state_history()`, forking, `update_state()` |
| 21 | Functional API: `@entrypoint`, `@task`, `previous` |
| 22 | Deep Agents: `create_deep_agent()`, planning, sub-agents, filesystem |
| 23 | LangChain 1.0 Utilities: `init_chat_model()`, `trim_messages()` |
| 24 | Agent Middleware: `before_model`, `after_model`, `create_agent()` |
| 25 | MCP + A2A: standardised tool and agent interoperability |
| 26 | Streaming: tokens, custom progress, and subgraph events |
| 27 | Fault tolerance: retries, timeouts, error handlers, graceful drain |
| 28 | Decisions: local decision model, model routing, risky-tool blocking |

Next steps:
1. Stream a run to a UI — Tutorial 26
2. Build a production multi-agent system combining your favourite patterns
3. Deploy with the open-source LangGraph server